# Notebook 06 — Importancia de Atributos y Análisis de Adjetivos

## Objetivo

Interpretar qué términos lingüísticos asocian los modelos lineales del Experimento 1 con fake vs real, comparar consistencia entre LR / LinearSVC / Multinomial NB, y validar la Hipótesis 1 sobre adjetivos emocionales/sensacionalistas.

Referencia metodológica: [ADR Experimento 4](../docs/adr/experimento-04-importancia-atributos.md).

> Los coeficientes reflejan patrones del dataset, no veracidad factual.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import spacy

from nlp.interpretability import (
    adjective_sentiment_summary,
    adjective_sentiment_table,
    best_config_rows_per_model,
    extract_adjective_counts,
    feature_importance_dataframe,
    fit_bigram_interpretation_pipeline,
    load_best_baseline_pipeline,
    load_source_normalization_flag,
    prepare_text_series,
    term_overlap_summary,
    top_bigrams_table,
    top_ngrams_by_class_frequency,
    top_terms_table,
)
from nlp.io import load_splits
from nlp.modeling import (
    baseline_text_columns,
    config_from_row,
    fit_pipeline_from_config,
)
from nlp.paths import (
    RESULTS_FIGURES,
    RESULTS_METRICS,
    RESULTS_MODELS,
    SOURCE_ABLATION_DECISION,
)
from nlp.plotting import (
    plot_adjective_frequency_comparison,
    plot_top_terms_horizontal,
    plot_wordcloud_from_frequencies,
    setup_style,
)

setup_style()
FIGURE_PREFIX = "06_"
TOP_K = 15

## 1. Carga de datos, ablación de fuente y mejor baseline (politics)

Se deriva la columna de texto desde la config del mejor modelo (no se fija `full_text` a mano). Si la ablación de fuente del notebook 03 activó normalización, se aplica a los análisis descriptivos.

In [2]:
baseline_results = pd.read_csv(RESULTS_METRICS / "baseline_results.csv")
pol_train, _, pol_test = load_splits("politics", columns=baseline_text_columns())

normalize_source = load_source_normalization_flag(SOURCE_ABLATION_DECISION)
print(
    "Normalización de fuente en análisis descriptivos:",
    "sí" if normalize_source else "no",
)

model_path = RESULTS_MODELS / "best_baseline_politics.joblib"
config_path = RESULTS_MODELS / "best_baseline_politics_config.json"

pipe, best_config, TEXT_COL = load_best_baseline_pipeline(
    model_path,
    config_path,
    baseline_results,
    pol_train,
    normalize_source=normalize_source,
)

print("Mejor baseline politics:")
print(
    pd.Series(best_config)[
        ["model", "vectorizer", "text_field", "stopwords", "ngram_range", "max_features"]
    ].to_string()
)
print(f"Columna de texto para interpretación: {TEXT_COL}")


Normalización de fuente en análisis descriptivos: no
Mejor baseline politics:
model                  linear_svm
vectorizer                    bow
text_field                   body
stopwords       without_stopwords
ngram_range                (1, 2)
max_features                10000
Columna de texto para interpretación: clean_body_text_without_stopwords


## 2. Comparación multi-modelo: LR, LinearSVC y Multinomial NB

Para cada algoritmo se toma su mejor config en validación (`baseline_results.csv`), se refitea en train y se extraen los top términos. El solapamiento Jaccard entre top-K mide si los patrones son consistentes entre modelos.

In [3]:
best_per_model = best_config_rows_per_model(baseline_results)
importance_tables: list[pd.DataFrame] = []
terms_fake_by_model: dict[str, set[str]] = {}
terms_real_by_model: dict[str, set[str]] = {}

for _, row in best_per_model.iterrows():
    model_name = row["model"]
    config = config_from_row(row)
    model_pipe, _ = fit_pipeline_from_config(
        pol_train,
        config,
        normalize_source=normalize_source,
    )
    importance = feature_importance_dataframe(model_pipe)
    table = top_terms_table(importance, n=30, model=model_name)
    importance_tables.append(table)

    terms_fake_by_model[model_name] = set(
        importance.nlargest(TOP_K, "coefficient")["term"]
    )
    terms_real_by_model[model_name] = set(
        importance.nsmallest(TOP_K, "coefficient")["term"]
    )

feature_importance = pd.concat(importance_tables, ignore_index=True)
feature_importance.to_csv(RESULTS_METRICS / "feature_importance.csv", index=False)

overlap_fake = term_overlap_summary(terms_fake_by_model)
overlap_real = term_overlap_summary(terms_real_by_model)
overlap_fake["direction"] = "fake"
overlap_real["direction"] = "real"
term_overlap = pd.concat([overlap_fake, overlap_real], ignore_index=True)
term_overlap.to_csv(RESULTS_METRICS / "feature_importance_overlap.csv", index=False)

print("Mejor config por modelo (val):")
print(
    best_per_model[
        ["model", "vectorizer", "text_field", "f2_fake", "ngram_range"]
    ].to_string(index=False)
)
print("\nSolapamiento Jaccard (top términos fake):")
print(overlap_fake.to_string(index=False))
print("\nSolapamiento Jaccard (top términos real):")
print(overlap_real.to_string(index=False))

best_importance = feature_importance_dataframe(pipe)
top_fake = best_importance.nlargest(30, "coefficient")[["term", "coefficient"]]
top_real = best_importance.nsmallest(30, "coefficient")[["term", "coefficient"]]
feature_importance.head(10)


Mejor config por modelo (val):
              model vectorizer text_field  f2_fake ngram_range
         linear_svm        bow       body 0.992366      (1, 2)
logistic_regression        bow       body 0.990522      (1, 2)
     multinomial_nb        bow      title 0.953036      (1, 2)

Solapamiento Jaccard (top términos fake):
            model_a             model_b  overlap  jaccard direction
         linear_svm          linear_svm       15      1.0      fake
         linear_svm logistic_regression       10      0.5      fake
         linear_svm      multinomial_nb        0      0.0      fake
logistic_regression logistic_regression       15      1.0      fake
logistic_regression      multinomial_nb        0      0.0      fake
     multinomial_nb      multinomial_nb       15      1.0      fake

Solapamiento Jaccard (top términos real):
            model_a             model_b  overlap  jaccard direction
         linear_svm          linear_svm       15     1.00      real
         linear_svm

,term,coefficient,direction,model
0,via,0.521961,fake,linear_svm
1,read,0.225246,fake,linear_svm
2,ipsos,0.181121,fake,linear_svm
3,polling,0.168768,fake,linear_svm
4,hillary,0.166019,fake,linear_svm
5,watch,0.164459,fake,linear_svm
6,cia,0.157691,fake,linear_svm
7,url,0.150380,fake,linear_svm
8,cia director,0.146205,fake,linear_svm
9,polls,0.138288,fake,linear_svm


In [4]:
plot_top_terms_horizontal(
    top_fake,
    top_real,
    n=TOP_K,
    save_path=RESULTS_FIGURES / f"{FIGURE_PREFIX}feature_importance_top_terms.png",
)
plt.show()


## 3. Bigramas predictivos vs frecuencia descriptiva

Los **bigramas con mayor peso** en el modelo (coeficientes de términos con espacio) responden al ADR. La frecuencia por clase es contexto descriptivo complementario.

In [5]:
bigram_pipe, _ = fit_bigram_interpretation_pipeline(
    pol_train,
    best_config,
    normalize_source=normalize_source,
)
bigram_importance = feature_importance_dataframe(bigram_pipe)
bigram_weights = top_bigrams_table(
    bigram_importance,
    n=20,
    model=best_config["model"],
)
bigram_weights.to_csv(RESULTS_METRICS / "bigram_importance.csv", index=False)

analysis_df = pd.concat([pol_train, pol_test], ignore_index=True)
analysis_text = prepare_text_series(
    analysis_df[TEXT_COL],
    normalize_source=normalize_source,
)
analysis_df = analysis_df.assign(_analysis_text=analysis_text)

fake_bi_freq = top_ngrams_by_class_frequency(
    analysis_df,
    "_analysis_text",
    1,
    normalize_source=False,
)
real_bi_freq = top_ngrams_by_class_frequency(
    analysis_df,
    "_analysis_text",
    0,
    normalize_source=False,
)

print("Top bigramas por PESO del modelo (fake):", bigram_weights[bigram_weights["direction"] == "fake"].head(10).to_dict("records"))
print("Top bigramas por PESO del modelo (real):", bigram_weights[bigram_weights["direction"] == "real"].head(10).to_dict("records"))
print("\nBigramas frecuentes FAKE (descriptivo):", fake_bi_freq[:10])
print("Bigramas frecuentes REAL (descriptivo):", real_bi_freq[:10])


Top bigramas por PESO del modelo (fake): [{'term': 'cia director', 'coefficient': 0.1462046195514572, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'told reuters', 'coefficient': 0.13504070260160717, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'reuters ipsos', 'coefficient': 0.124121227269644, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'twitter com', 'coefficient': 0.12050562739366497, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'breitbart news', 'coefficient': 0.1089204816740368, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'pic twitter', 'coefficient': 0.09853609964272755, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'anyone else', 'coefficient': 0.08655849472353307, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'obama administration', 'coefficient': 0.07999733218778804, 'direction': 'fake', 'model': 'linear_svm'}, {'term': 'president trump', 'coefficient': 0.07403396306613337, 'direction': 'fake', 'model': 'linear_svm'

In [6]:
top_fake_bi = bigram_weights[bigram_weights["direction"] == "fake"].head(TOP_K)
top_real_bi = bigram_weights[bigram_weights["direction"] == "real"].head(TOP_K)

plot_top_terms_horizontal(
    top_fake_bi,
    top_real_bi,
    n=TOP_K,
    save_path=RESULTS_FIGURES / f"{FIGURE_PREFIX}bigram_importance_top_terms.png",
)
plt.show()

## 4. Adjetivos con spaCy y carga semántica (VADER)

Validación de la Hipótesis 1: adjetivos con mayor carga emocional en fake. Se usa la misma columna que el modelo (`body`) y train+test para la lectura cualitativa (vocabulario aprendido en train; ejemplos de test incluidos).

In [7]:
nlp = spacy.load("en_core_web_sm")

adj_analysis_df = analysis_df[[ "label", "_analysis_text"]].copy()
fake_texts = adj_analysis_df.loc[adj_analysis_df["label"] == 1, "_analysis_text"]
real_texts = adj_analysis_df.loc[adj_analysis_df["label"] == 0, "_analysis_text"]

fake_adj = extract_adjective_counts(nlp, fake_texts, sample_size=3000)
real_adj = extract_adjective_counts(nlp, real_texts, sample_size=3000)

fake_adj_df = adjective_sentiment_table(fake_adj, class_label="fake")
real_adj_df = adjective_sentiment_table(real_adj, class_label="real")
adj_df = pd.concat([fake_adj_df, real_adj_df], ignore_index=True)
adj_df.to_csv(RESULTS_METRICS / "adjectives_by_class.csv", index=False)

adj_summary = adjective_sentiment_summary(adj_df)
adj_summary.to_csv(RESULTS_METRICS / "adjectives_sentiment_summary.csv", index=False)

print("Resumen de carga emocional (VADER, promedio ponderado por frecuencia):")
print(adj_summary.to_string(index=False))
adj_df.head(10)


Resumen de carga emocional (VADER, promedio ponderado por frecuencia):
class  n_adjectives  total_count  mean_vader_abs_weighted  mean_vader_compound_weighted
 fake            50        24502                  0.06949                      0.014727
 real            50        37495                  0.02727                      0.010174


,adjective,count,vader_compound,vader_abs,class
0,black,1295,0.0000,0.0000,fake
1,many,1001,0.0000,0.0000,fake
2,last,970,0.0000,0.0000,fake
3,american,917,0.0000,0.0000,fake
4,new,834,0.0000,0.0000,fake
5,presidential,783,0.0000,0.0000,fake
6,political,756,0.0000,0.0000,fake
7,good,749,0.4404,0.4404,fake
8,first,727,0.0000,0.0000,fake
9,former,715,0.0000,0.0000,fake


In [8]:
plot_adjective_frequency_comparison(
    fake_adj,
    real_adj,
    top_n=TOP_K,
    save_path=RESULTS_FIGURES / f"{FIGURE_PREFIX}adjectives_by_class.png",
)
plt.show()

for class_label, counter, colormap in [
    ("fake", fake_adj, "Reds"),
    ("real", real_adj, "Greens"),
]:
    fig = plot_wordcloud_from_frequencies(
        counter,
        title=f"Adjetivos — {class_label}",
        colormap=colormap,
        save_path=RESULTS_FIGURES / f"{FIGURE_PREFIX}adjectives_wordcloud_{class_label}.png",
    )
    if fig is not None:
        plt.show()
    else:
        print("wordcloud no instalado; se omite word cloud de adjetivos")


## 5. Conclusiones

- **Términos y bigramas predictivos:** los coeficientes del mejor baseline y los bigramas con mayor peso muestran qué expresiones el modelo asocia con fake (p. ej. lenguaje sensacionalista, CTAs como *watch/read*) vs real (marcas de agencia y estilo periodístico).
- **Consistencia multi-modelo:** el solapamiento Jaccard entre LR, LinearSVC y Multinomial NB indica si los patrones discriminativos son robustos al algoritmo o dependen de una sola familia de modelos.
- **Hipótesis 1 (adjetivos):** la comparación de frecuencias y el promedio ponderado de `|vader_compound|` por clase permiten contrastar si fake usa adjetivos con mayor carga emocional; los coeficientes de features lingüísticas del notebook 04 (`sentimiento_vader`, `ratio_exclamacion`) complementan esta lectura.
- **Limitación:** estas asociaciones son correlaciones estilísticas del corpus político; no implican verificación factual ni aplican a DistilBERT (caja negra → Experimento 5).